In [0]:
dbutils.widgets.text("catalog","fk_1361")
catalog = dbutils.widgets.get("catalog")

In [0]:
%sql
select * from fk_1361.silver.customers_silver limit 5

In [0]:
spark.sql(f"""
create table if not exists {catalog}.silver.customers_scd as
select customer_id, full_name, email, phone, city, tier, signup_date, cast(null as date) as effective_from, 
cast(null as date) as effective_to,cast(null as boolean) as is_current  from {catalog}.silver.customers_silver
where 1=0
""")

In [0]:
spark.sql(f"""
merge into {catalog}.silver.customers_scd t 
using (
    select customer_id, full_name, email, phone, city, tier, signup_date from {catalog}.silver.customers_silver
    qualify row_number() over (partition by customer_id order by signup_date desc) = 1
) s
on t.customer_id = s.customer_id and t.is_current = true
when matched and t.full_name <> s.full_name or t.email <> s.email or t.phone <> s.phone or t.city <> s.city or t.tier <> s.tier or t.signup_date <> s.signup_date then update set
t.effective_to = current_date(), t.is_current = false
when not matched then insert (
    customer_id, full_name, email, phone, city, tier, signup_date, effective_from, effective_to, is_current
) values (
    s.customer_id, s.full_name, s.email, s.phone, s.city, s.tier, s.signup_date, current_date(), null, true
)
""")

spark.sql(f"""
insert into {catalog}.silver.customers_scd (
customer_id, full_name, email, phone, city, tier, signup_date, effective_from, effective_to, is_current
) 
select customer_id, full_name, email, phone, city, tier, signup_date, current_date(), null, true from {catalog}.silver.customers_silver
""")